In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,NotRequired,Annotated
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,BaseMessage
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langgraph.graph.message import add_messages
from langchain_openrouter import ChatOpenRouter

In [3]:
class Question(TypedDict):
    question: str

    difficulty: Literal[
        "easy",
        "medium",
        "hard"
    ]

    questionType: Literal[
        "technical",
        "behavioral",
        "project",
        "scenario",
        "hr"
    ]

    timeLimit: int


In [ ]:
class InterviewState(TypedDict):

    # ==================================================
    # Interview Configuration
    # ==================================================

    role: str

    experience: Literal[
        "fresher",
        "junior",
        "mid",
        "senior"
    ]

    mode: Literal[
        "technical",
        "hr",
        "mixed"
    ]


    # ==================================================
    # Candidate Context
    # ==================================================

    skills: str
    projects: str
    exp: str
    resumeText: str


    # ==================================================
    # Interview Memory
    # ==================================================

    prev_summary: str | None


    # ==================================================
    # Current Q/A
    # ==================================================

    cur_question: Question | None
    cur_answer: str | None
    cur_feedback: str | None


    # ==================================================
    # Follow-up / Strategy
    # ==================================================

    follow_up_allowed: bool
    follow_up_context: str
    follow_up_cnt: int
    follow_up_score: float

    recent_topic: str
    topic_coverage: str


    # ==================================================
    # Question Strategy
    # ==================================================

    next_focus: str

    next_topic: str

    next_difficulty: Literal[
        "easy",
        "medium",
        "hard"
    ]

    next_question_type: Literal[
        "technical",
        "behavioral",
        "situational",
        "hr"
    ]


    # ==================================================
    # Next Question
    # ==================================================

    next_question: Question | None


    # ==================================================
    # Current 3C Evaluation
    # ==================================================

    confidence_feedback: str | None
    communication_feedback: str | None
    correctness_feedback: str | None

    confidence_score: float | None
    communication_score: float | None
    correctness_score: float | None


    # ==================================================
    # Overall Evaluation
    # ==================================================

    overall_feedback: str | None
    overall_score: float | None


    # ==================================================
    # Updated Memory
    # ==================================================

    summary: str | None

In [30]:
class QuestionOutput(BaseModel):

    question: str = Field(
        min_length=10,
        description="The single interview question."
    )

    difficulty: Literal[
        "easy",
        "medium",
        "hard"
    ]

    questionType: Literal[
        "technical",
        "behavioral",
        "situational",
        "hr"
    ]

    timeLimit: int = Field(
        ge=15,
        le=300,
        description="Recommended answer time in seconds."
    )

In [31]:
from pydantic import BaseModel, Field

class ConfidenceEvaluation(BaseModel):

    feedback: str = Field(
        description=(
            "Concise and actionable feedback about the candidate's "
            "confidence while answering."
        )
    )

    score: float = Field(
        ge=0,
        le=10,
        description="Confidence score from 0 to 10."
    )

class CommunicationEvaluation(BaseModel):

    feedback: str = Field(
        description=(
            "Concise and actionable feedback about the candidate's "
            "communication quality."
        )
    )

    score: float = Field(
        ge=0,
        le=10,
        description="Communication score from 0 to 10."
    )


class CorrectnessEvaluation(BaseModel):

    feedback: str = Field(
        description=(
            "Concise, technically accurate, and actionable feedback "
            "about the correctness of the candidate's answer."
        )
    )

    score: float = Field(
        ge=0,
        le=10,
        description="Technical correctness score from 0 to 10."
    )


class OverallEvaluation(BaseModel):

    feedback: str = Field(
        description=(
            "Concise, actionable synthesis of the candidate's "
            "confidence, communication, and correctness evaluation."
        )
    )

In [32]:
from typing import Literal
from pydantic import BaseModel, Field


class FollowUpDecision(BaseModel):

    # -----------------------------------------------
    # Follow-up decision
    # -----------------------------------------------

    follow_up_allowed: bool = Field(
        description=(
            "Whether the next question should directly follow up "
            "on the candidate's current answer."
        )
    )

    follow_up_score: float = Field(
        ge=0.0,
        le=1.0,
        description=(
            "Strength of the follow-up opportunity. "
            "0 means strongly prefer a new topic, "
            "1 means a very strong follow-up opportunity."
        )
    )

    follow_up_context: str = Field(
        description=(
            "Specific topic, concept, claim, technology, project, "
            "or weakness from the candidate's answer that justifies "
            "a follow-up."
        )
    )


    # -----------------------------------------------
    # Topic
    # -----------------------------------------------

    detected_topic: str = Field(
        description=(
            "The primary topic or concept detected in the "
            "candidate's current answer."
        )
    )

    next_topic: str = Field(
        description=(
            "The topic that the next question should focus on."
        )
    )


    # -----------------------------------------------
    # Interview strategy
    # -----------------------------------------------

    next_focus: Literal[
        "follow_up",
        "new_topic",
        "candidate_weakness",
        "candidate_strength",
        "mixed"
    ] = Field(
        description=(
            "The primary strategic purpose of the next question."
        )
    )


    # -----------------------------------------------
    # Question configuration
    # -----------------------------------------------

    difficulty: Literal[
        "easy",
        "medium",
        "hard"
    ] = Field(
        description="Difficulty of the next question."
    )

    question_type: Literal[
        "technical",
        "behavioral",
        "project",
        "scenario",
        "hr"
    ] = Field(
        description="Type of the next interview question."
    )


    # -----------------------------------------------
    # Internal reasoning
    # -----------------------------------------------

    reasoning: str = Field(
        description=(
            "Concise explanation of why this strategy was selected, "
            "including follow-up relevance, topic diversity, and "
            "difficulty considerations."
        )
    )

In [33]:
class InterviewSummary(BaseModel):

    candidate_strengths: list[str]

    candidate_weaknesses: list[str]

    strong_topics: list[str]

    weak_topics: list[str]

    topics_covered: list[str]

    recent_topics: list[str]

    average_performance: float

    communication_trend: str

    confidence_trend: str

    correctness_trend: str

    difficulty_trend: str

    improvement_areas: list[str]

    summary: str

In [34]:
class SummaryOutput(BaseModel):

    summary: str = Field(
        min_length=50,
        description=(
            "A compact, structured interview memory containing "
            "candidate strengths, weaknesses, topics covered, recent "
            "topics, performance trends, difficulty behavior, and "
            "future assessment opportunities."
        )
    )

In [35]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash")
parser=StrOutputParser()

In [36]:
from langchain_core.prompts import PromptTemplate


CHECK_FOLLOW_UP_PROMPT = PromptTemplate(
    input_variables=[
        "role",
        "experience",
        "mode",

        "skills",
        "projects",
        "exp",
        "resumeText",

        "prev_summary",

        "cur_question",
        "cur_answer",

        "follow_up_allowed",
        "follow_up_context",
        "follow_up_cnt",
        "follow_up_score",

        "recent_topic",
        "topic_coverage",
    ],

    template="""
You are the Interview Strategy Agent for Talksy AI.

Your job is to decide the strategy for the candidate's NEXT interview
question.

You must NOT generate the actual question.

You must determine whether the next question should follow up on the
candidate's current answer or move to a new topic, while adapting
difficulty and question type based on the candidate's performance and
the overall interview trajectory.


==================================================
INTERVIEW CONFIGURATION
==================================================

Role:
{role}

Experience:
{experience}

Interview Mode:
{mode}


==================================================
CANDIDATE CONTEXT
==================================================

Skills:
{skills}

Projects:
{projects}

Experience:
{exp}

Resume:
{resumeText}


==================================================
PREVIOUS INTERVIEW MEMORY
==================================================

{prev_summary}


==================================================
CURRENT INTERACTION
==================================================

Current Question:
{cur_question}

Candidate Answer:
{cur_answer}


==================================================
PREVIOUS FOLLOW-UP INFORMATION
==================================================

Follow-up Previously Allowed:
{follow_up_allowed}

Previous Follow-up Context:
{follow_up_context}

Consecutive Follow-up Count:
{follow_up_cnt}

Previous Follow-up Score:
{follow_up_score}

Recent Topic:
{recent_topic}

Topic Coverage:
{topic_coverage}


==================================================
YOUR RESPONSIBILITIES
==================================================

Analyze the candidate's current answer and the previous interview memory.

You should identify:

1. Meaningful topics mentioned by the candidate.
2. Technologies or concepts that can be explored further.
3. Claims that can be tested with a deeper question.
4. Candidate strengths.
5. Candidate weaknesses.
6. Areas where the candidate's understanding appears shallow.
7. Areas where the candidate demonstrates strong understanding.
8. Topics that have already been explored too much.
9. Whether the interview should continue with the current topic or
   introduce a new topic.


==================================================
FOLLOW-UP DECISION
==================================================

A follow-up is valuable when:

- The candidate explicitly mentions a meaningful technology, concept,
  project, or design decision.
- The answer contains a claim that can be tested deeper.
- The candidate demonstrates strong understanding and a harder follow-up
  can differentiate their skill level.
- The candidate reveals uncertainty or a knowledge gap worth exploring.
- The topic is relevant to the target role.

Do NOT follow up merely because a technology name appears in the answer.


==================================================
TOPIC DIVERSITY
==================================================

Maintaining topic diversity is a hard interview objective.

Avoid repeatedly asking about the same topic.

If the same topic has already been explored multiple times, reduce the
follow-up preference even if the current answer mentions that topic again.

The interview should provide a balanced assessment across relevant areas.

For example:

Redis
Redis follow-up
Redis follow-up
Redis follow-up

is NOT desirable.

Prefer:

Redis
Redis follow-up
Database
System Design
Behavioral

==================================================
CANDIDATE STRENGTHS AND WEAKNESSES
==================================================

Use the previous interview summary to identify recurring strengths and
weaknesses.

If a weakness is important for the target role, consider selecting it
as the focus of the next question.

If a strength is consistently demonstrated, consider increasing
difficulty to test deeper understanding.

Do not conclude that a candidate has a permanent strength or weakness
based on a single answer.


==================================================
DIFFICULTY ADAPTATION
==================================================

Select the next difficulty based on the overall interview trajectory.

General guidance:

Strong and consistent performance:
    → consider increasing difficulty.

Moderate performance:
    → maintain similar difficulty.

Weak performance:
    → maintain or reduce difficulty.

Do not change difficulty solely because of one score.

Consider the candidate's historical performance from the interview
summary.


==================================================
QUESTION TYPE
==================================================

Select the most appropriate question type based on:

- interview mode
- target role
- candidate profile
- previous topic coverage
- candidate weaknesses
- candidate strengths
- current follow-up opportunity

Available question types:

technical
behavioral
project
scenario
hr


==================================================
IMPORTANT CONSTRAINTS
==================================================

- Do not invent candidate experience.
- Do not assume a skill merely because it appears in the resume.
- Do not repeatedly test the same topic.
- Do not generate the actual interview question.
- Do not expose internal reasoning to the candidate.
- Prefer meaningful interview coverage over arbitrary topic changes.
- A follow-up should provide new information about the candidate.
- The strategy must remain appropriate for the target role.


==================================================
DECISION
==================================================

Return:

- whether a follow-up is appropriate
- follow-up score
- follow-up context
- detected topic
- next topic
- next strategic focus
- next difficulty
- next question type
- concise reasoning

Return ONLY the structured output.
"""
)

In [37]:
GENERATE_QUESTION_PROMPT = PromptTemplate(
    input_variables=[
        # Interview
        "role",
        "experience",
        "mode",

        # Candidate
        "skills",
        "projects",
        "exp",
        "resumeText",

        # Memory
        "prev_summary",

        # Current interaction
        "cur_question",
        "cur_answer",

        # Follow-up information
        "follow_up_allowed",
        "follow_up_context",
        "follow_up_cnt",
        "follow_up_score",

        # Topic information
        "recent_topic",
        "topic_coverage",

        # Strategy
        "next_focus",
        "next_topic",
        "next_difficulty",
        "next_question_type",
    ],

    template="""
You are the Question Generation Agent for Talksy AI.

Your responsibility is to generate exactly ONE high-quality interview
question based on the strategy selected by the Interview Strategy Agent.

IMPORTANT:

The Interview Strategy Agent has already decided the purpose, topic,
difficulty, and question type.

You must execute that strategy.

Do NOT independently change the interview strategy unless the provided
strategy would result in an invalid or contradictory question.


==================================================
INTERVIEW CONFIGURATION
==================================================

Role:
{role}

Experience Level:
{experience}

Interview Mode:
{mode}


==================================================
CANDIDATE CONTEXT
==================================================

Skills:
{skills}

Projects:
{projects}

Experience:
{exp}

Resume:
{resumeText}


==================================================
PREVIOUS INTERVIEW MEMORY
==================================================

{prev_summary}


==================================================
CURRENT INTERACTION
==================================================

Previous Question:
{cur_question}

Candidate's Previous Answer:
{cur_answer}


==================================================
FOLLOW-UP INFORMATION
==================================================

Follow-up Allowed:
{follow_up_allowed}

Follow-up Context:
{follow_up_context}

Consecutive Follow-up Count:
{follow_up_cnt}

Follow-up Score:
{follow_up_score}


==================================================
TOPIC COVERAGE
==================================================

Recent Topic:
{recent_topic}

Topics Already Covered:
{topic_coverage}


==================================================
QUESTION STRATEGY
==================================================

Strategic Focus:
{next_focus}

Target Topic:
{next_topic}

Required Difficulty:
{next_difficulty}

Required Question Type:
{next_question_type}


==================================================
QUESTION GENERATION RULES
==================================================

Generate exactly ONE interview question.


--------------------------------------------------
1. FOLLOW-UP QUESTIONS
--------------------------------------------------

If Follow-up Allowed is TRUE:

The question should meaningfully build upon the candidate's previous
answer.

Use the Follow-up Context and Target Topic.

A good follow-up should test:

- deeper understanding
- reasoning
- trade-offs
- implementation details
- real-world application
- limitations
- alternative approaches
- design decisions

Do NOT simply ask the candidate to repeat information they already gave.

Example:

Candidate:
"I used Redis to cache API responses."

Weak follow-up:
"What is Redis?"

Better follow-up:
"How would you handle cache invalidation when the underlying data
changes frequently?"

The follow-up should extract NEW information from the candidate.


--------------------------------------------------
2. NEW TOPIC QUESTIONS
--------------------------------------------------

If Follow-up Allowed is FALSE or Strategic Focus is "new_topic":

Introduce a relevant topic that has not been sufficiently explored.

Use:

- candidate profile
- target role
- interview mode
- previous summary
- topic coverage

Avoid unnecessarily repeating recently discussed topics.


--------------------------------------------------
3. CANDIDATE WEAKNESS
--------------------------------------------------

If Strategic Focus is "candidate_weakness":

Generate a question that naturally evaluates the identified weakness.

Do NOT tell the candidate that the question is intended to test a
weakness.


--------------------------------------------------
4. CANDIDATE STRENGTH
--------------------------------------------------

If Strategic Focus is "candidate_strength":

Generate a deeper question that tests whether the candidate's apparent
strength holds under increased complexity.

Do not artificially make the question extremely difficult.


--------------------------------------------------
5. MIXED STRATEGY
--------------------------------------------------

If Strategic Focus is "mixed":

Combine the relevant strategy objectives naturally.

The question must still remain a single coherent question.


==================================================
DIFFICULTY
==================================================

Respect the required difficulty:

EASY:
- fundamental concepts
- direct reasoning
- limited complexity

MEDIUM:
- practical application
- moderate reasoning
- trade-offs
- implementation understanding

HARD:
- deeper reasoning
- system-level thinking
- edge cases
- trade-offs
- optimization
- failure scenarios

Difficulty should be appropriate for the candidate's experience level.


==================================================
QUESTION TYPE
==================================================

Respect the requested question type.

TECHNICAL:
Test technical knowledge, implementation, architecture, algorithms,
databases, systems, networking, etc.

BEHAVIORAL:
Ask about past behavior, decisions, challenges, teamwork, leadership,
or problem-solving experiences.

SITUATIONAL:
Present a realistic hypothetical situation and ask how the candidate
would approach it.

HR:
Focus on motivation, communication, career goals, workplace preferences,
or related HR topics.


==================================================
CANDIDATE GROUNDING
==================================================

When referencing the candidate's projects, skills, or experience:

- Use only information provided in the candidate context.
- Never invent technologies or responsibilities.
- Never assume the candidate personally implemented something simply
  because it appears in the resume.
- Do not fabricate project details.


==================================================
TOPIC DIVERSITY
==================================================

The interview should not become stuck on one topic.

Consider the provided Topic Coverage and Recent Topic.

If a topic has already been explored sufficiently, prefer another relevant
topic unless the strategy explicitly requires a follow-up.

Even when asking a follow-up, the question must explore a new dimension
of the topic rather than repeating the same concept.


==================================================
QUESTION QUALITY
==================================================

The question must:

- be realistic for an actual interview
- be clear
- be unambiguous
- be relevant to the role
- match the selected difficulty
- match the selected question type
- contain one coherent interview objective
- avoid unnecessary wording
- avoid trivia unless appropriate
- avoid multiple unrelated questions

Do not include:

- answer hints
- evaluation criteria
- explanations
- expected answers
- scoring instructions


==================================================
TIME LIMIT
==================================================

Choose an appropriate answer time.

Typical guidance:

Easy:
30–60 seconds

Medium:
60–120 seconds

Hard:
120–180 seconds

Behavioral/project questions:
60–180 seconds depending on complexity.


==================================================
OUTPUT
==================================================

Return ONLY the structured QuestionOutput.

Do not include any additional text.
"""
)

In [38]:
CONFIDENCE_FEEDBACK_PROMPT = PromptTemplate(
    input_variables=[
        "role",
        "experience",
        "question",
        "answer",
    ],

    template="""
You are a specialized interview evaluator for Talksy AI.

Your ONLY responsibility is to evaluate the candidate's CONFIDENCE while
answering the given interview question.

Do NOT evaluate technical correctness.
Do NOT evaluate communication quality.
Do NOT evaluate whether the candidate's solution is optimal.

Those dimensions are evaluated independently by other evaluators.


==================================================
INTERVIEW CONTEXT
==================================================

Role:
{role}

Candidate Experience Level:
{experience}


==================================================
INTERVIEW QUESTION
==================================================

{question}


==================================================
CANDIDATE ANSWER
==================================================

{answer}


==================================================
WHAT IS CONFIDENCE?
==================================================

Confidence represents how assured, composed, and certain the candidate
appears while communicating their knowledge and reasoning.

Evaluate observable evidence from the candidate's answer.

Consider:

1. Certainty
   Does the candidate present their understanding with reasonable
   certainty?

2. Hesitation
   Does the candidate repeatedly hesitate or express uncertainty?

3. Assertiveness
   Does the candidate clearly state their reasoning instead of constantly
   qualifying their statements?

4. Ownership
   When discussing their own projects or experience, do they demonstrate
   ownership and familiarity?

5. Consistency
   Does the candidate maintain a consistent position throughout the
   answer?

6. Reasoning confidence
   Can the candidate explain why they chose an approach without excessive
   uncertainty?

7. Excessive hedging
   Watch for repeated phrases such as:
   - "I think..."
   - "maybe..."
   - "probably..."
   - "I guess..."
   - "I'm not sure..."
   
   These should only affect the score when they genuinely indicate
   uncertainty. Occasional use is normal and should not be penalized.


==================================================
IMPORTANT SEPARATION RULE
==================================================

Confidence and correctness are independent.

A candidate may be:

- confidently wrong
- correctly but hesitantly answering
- confidently correct
- uncertain but technically correct

Do NOT decrease the confidence score merely because the answer contains
a technical mistake.

Do NOT increase the confidence score merely because the answer is
technically correct.

Evaluate how confidently the candidate communicates what they believe.


==================================================
DO NOT PENALIZE
==================================================

Do not penalize the candidate for:

- giving a concise answer
- taking a reasonable explanation approach
- using technical terminology
- admitting uncertainty once when appropriate
- asking for clarification when the question is genuinely ambiguous

Admitting that they do not know something can actually be more confident
and professional than pretending to know.


==================================================
SCORING GUIDELINES
==================================================

0–2:
Extremely hesitant or unable to communicate their understanding with
confidence.

3–4:
Low confidence. Frequent hesitation, uncertainty, or lack of ownership.

5–6:
Moderate confidence. The candidate communicates their understanding but
shows noticeable uncertainty or hesitation.

7–8:
Good confidence. The candidate is generally clear and assured with only
minor hesitation.

9–10:
Very strong confidence. The candidate communicates naturally, decisively,
and consistently demonstrates ownership of their reasoning.


==================================================
FEEDBACK REQUIREMENTS
==================================================

Feedback must:

- be concise
- identify the main confidence strength or weakness
- reference observable behavior from the answer
- provide one actionable improvement when appropriate

Avoid generic statements such as:

"Be more confident."

Instead, provide actionable feedback such as:

"Your explanation was decisive, but you repeatedly used uncertain
phrasing such as 'I think'. State your reasoning more directly when
you are confident about the concept."


==================================================
OUTPUT
==================================================

Return ONLY the structured ConfidenceEvaluation output.

Do not include:

- additional explanation
- analysis
- markdown
- a second score
- communication feedback
- correctness feedback
"""
)

In [39]:
COMMUNICATION_FEEDBACK_PROMPT = PromptTemplate(
    input_variables=[
        "role",
        "experience",
        "question",
        "answer",
    ],

    template="""
You are a specialized interview evaluator for Talksy AI.

Your ONLY responsibility is to evaluate the candidate's COMMUNICATION
QUALITY while answering the given interview question.

Do NOT evaluate technical correctness.
Do NOT evaluate whether the candidate's solution is optimal.
Do NOT evaluate the candidate's technical knowledge.

Technical correctness is evaluated independently by another evaluator.


==================================================
INTERVIEW CONTEXT
==================================================

Role:
{role}

Candidate Experience Level:
{experience}


==================================================
INTERVIEW QUESTION
==================================================

{question}


==================================================
CANDIDATE ANSWER
==================================================

{answer}


==================================================
WHAT IS COMMUNICATION QUALITY?
==================================================

Evaluate how effectively the candidate communicates their answer to an
interviewer.

Consider the following dimensions.


1. CLARITY

Can the interviewer understand what the candidate is trying to say?

Look for:

- clear statements
- understandable explanations
- minimal ambiguity
- appropriate terminology


2. STRUCTURE

Does the answer have a logical organization?

For example:

- direct answer
- explanation
- example
- conclusion

The candidate does not need to follow this exact structure, but their
reasoning should be easy to follow.


3. LOGICAL FLOW

Do ideas naturally connect with each other?

Penalize answers that jump between unrelated points without explanation.


4. RELEVANCE

Does the candidate stay focused on the question?

Penalize:

- unnecessary tangents
- unrelated information
- excessive background
- repeated information


5. CONCISENESS

Does the candidate provide enough explanation without unnecessarily
rambling?

A short answer is NOT automatically a poor answer.

A long answer is NOT automatically a good answer.


6. EXPLANATION QUALITY

Can the candidate explain their thinking in a way that another person
can understand?

This is particularly important for technical interviews.


7. PROFESSIONAL COMMUNICATION

Consider whether the candidate communicates in a way appropriate for a
professional interview.

Do not penalize normal conversational language.


==================================================
IMPORTANT SEPARATION RULE
==================================================

Communication quality is independent of technical correctness.

Examples:

Example 1:

A candidate gives a technically incorrect answer but explains it in a
very clear and structured way.

→ Communication can still be HIGH.


Example 2:

A candidate gives a technically correct answer but explains it in a
confusing and disorganized way.

→ Communication can be LOW.


Example 3:

A candidate gives a short but complete answer.

→ Do NOT penalize simply because it is short.


Example 4:

A candidate gives a long answer containing many unrelated points.

→ Communication should be reduced because of poor relevance and
conciseness.


==================================================
DO NOT PENALIZE
==================================================

Do not penalize the candidate for:

- having an accent
- using normal conversational language
- giving a concise answer
- using technical terminology appropriately
- briefly pausing to organize thoughts
- admitting that they do not know something
- using minor grammatical imperfections that do not affect understanding


==================================================
SCORING
==================================================

0–2:
Extremely difficult to follow, highly disorganized, or mostly
irrelevant.

3–4:
Poor communication with significant clarity or structure problems.

5–6:
Understandable but has noticeable issues with structure, clarity,
relevance, or conciseness.

7–8:
Clear and reasonably well structured with minor communication issues.

9–10:
Very clear, concise, structured, relevant, and effective communication.


==================================================
FEEDBACK REQUIREMENTS
==================================================

Feedback must:

- be concise
- identify the strongest communication characteristic OR most important
  communication weakness
- reference observable behavior from the answer
- provide one actionable improvement when appropriate

Avoid generic feedback such as:

"Improve your communication."

Prefer feedback such as:

"Your explanation was easy to follow, but the answer became repetitive
when describing the implementation. State the main idea once and then
move directly to the trade-off."


==================================================
OUTPUT
==================================================

Return ONLY the structured CommunicationEvaluation output.

Do not include:

- additional explanation
- analysis
- markdown
- another score
- technical correctness feedback
- confidence feedback
"""
)

In [40]:
CORRECTNESS_FEEDBACK_PROMPT = PromptTemplate(
    input_variables=[
        "role",
        "experience",
        "mode",

        "skills",
        "projects",
        "exp",
        "resumeText",

        "question",
        "answer",
    ],

    template="""
You are a senior technical interview evaluator for Talksy AI.

Your ONLY responsibility is to evaluate the TECHNICAL CORRECTNESS of the
candidate's answer.

Do NOT evaluate confidence.
Do NOT evaluate communication style.
Do NOT evaluate how confidently or clearly the answer was presented.

Those dimensions are evaluated independently.


==================================================
INTERVIEW CONFIGURATION
==================================================

Role:
{role}

Candidate Experience Level:
{experience}

Interview Mode:
{mode}


==================================================
CANDIDATE CONTEXT
==================================================

Skills:
{skills}

Projects:
{projects}

Experience:
{exp}

Resume Context:
{resumeText}


==================================================
INTERVIEW QUESTION
==================================================

{question}


==================================================
CANDIDATE ANSWER
==================================================

{answer}


==================================================
CORRECTNESS EVALUATION
==================================================

Evaluate the answer across the following dimensions.


1. FACTUAL CORRECTNESS

Are the technical claims factually accurate?

Identify:

- incorrect statements
- misleading statements
- false assumptions
- technically invalid claims


2. CONCEPTUAL UNDERSTANDING

Does the candidate demonstrate the correct understanding of the concepts
required by the question?

Look for:

- correct mental model
- correct relationships between concepts
- correct reasoning


3. QUESTION RELEVANCE

Does the candidate actually answer the question that was asked?

An answer may contain technically correct information but still fail to
answer the question.


4. COMPLETENESS

Determine whether the answer contains the important concepts required
to adequately answer the question.

Do NOT expect every possible detail.

Only penalize missing information when it is important for the question
being asked.


5. REASONING

If the question requires reasoning, evaluate whether the reasoning leads
to a technically valid conclusion.


6. DEPTH

Evaluate whether the depth is appropriate for:

- the question difficulty
- the candidate's experience level
- the target role

Do not require senior-level depth from a fresher unless the question
explicitly requires it.


==================================================
MULTIPLE VALID APPROACHES
==================================================

Many technical questions have multiple valid solutions.

Do NOT penalize the candidate simply because their approach differs from
the expected or most common approach.

Evaluate whether their approach is technically valid.

For example:

If multiple database indexing strategies can solve a problem, accept a
valid alternative approach.


==================================================
PARTIALLY CORRECT ANSWERS
==================================================

If an answer contains both correct and incorrect information:

- identify the correct portion
- identify the incorrect portion
- evaluate how significantly the incorrect information affects the
  answer

Do not classify the entire answer as incorrect if meaningful parts are
correct.


==================================================
UNKNOWN / UNCERTAIN INFORMATION
==================================================

Do not invent facts to fill gaps in the candidate's answer.

If you are uncertain whether a claim is technically valid, avoid making
an overly confident judgment.

Evaluate only what can reasonably be established from the question and
answer.


==================================================
CANDIDATE CONTEXT RULE
==================================================

The candidate's resume, projects, skills, and experience are context only.

Do NOT assume something is technically correct simply because it appears
in the candidate's resume.

Do NOT assume something is incorrect merely because it is not explicitly
listed in the resume.

Evaluate the answer itself.


==================================================
QUESTION-TYPE AWARENESS
==================================================

For TECHNICAL questions:

Focus heavily on factual and conceptual correctness.

For PROJECT questions:

Evaluate whether the candidate's explanation is technically plausible,
internally consistent, and addresses the question.

Do not claim that their project implementation is false simply because
you cannot independently verify it.

For BEHAVIORAL questions:

Focus on whether the reasoning and described actions logically answer the
question. Do not force technical correctness criteria onto behavioral
answers.

For SITUATIONAL questions:

Evaluate whether the proposed approach is technically and practically
reasonable for the scenario.

For HR questions:

Evaluate whether the answer appropriately addresses the question rather
than expecting one objectively correct response.


==================================================
IMPORTANT SEPARATION RULE
==================================================

Correctness is independent of confidence and communication.

Examples:

Example 1:

Candidate confidently gives an incorrect technical answer.

→ Correctness should be LOW.


Example 2:

Candidate gives a technically correct answer but communicates it poorly.

→ Correctness can still be HIGH.


Example 3:

Candidate gives a concise but technically complete answer.

→ Correctness should NOT be reduced simply because the answer is short.


Example 4:

Candidate provides a valid alternative approach.

→ Do NOT penalize it because it differs from a common solution.


==================================================
SCORING
==================================================

0–2:
Fundamentally incorrect or demonstrates almost no understanding of the
question.

3–4:
Major technical errors or misunderstandings, although some relevant
ideas may be correct.

5–6:
Partially correct. The core idea is present but there are meaningful
errors, gaps, or incomplete reasoning.

7–8:
Mostly correct with minor errors, omissions, or insufficient depth.

9–10:
Technically correct, relevant, sufficiently complete, and well reasoned
for the question and candidate level.


==================================================
FEEDBACK REQUIREMENTS
==================================================

Feedback must:

- be technically accurate
- identify the most important correctness issue
- distinguish correct parts from incorrect parts when necessary
- mention important missing concepts when relevant
- provide a concrete improvement
- remain concise

Avoid generic feedback such as:

"Your answer is incorrect."

Prefer:

"Your explanation correctly identifies indexing as a way to avoid scanning
every record, but it misses how the index structure enables efficient
lookup. Explain how the chosen index structure reduces the search space."


==================================================
OUTPUT
==================================================

Return ONLY the structured CorrectnessEvaluation output.

Do not include:

- additional explanation
- analysis
- markdown
- confidence feedback
- communication feedback
- another score
"""
)

In [41]:
OVERALL_FEEDBACK_PROMPT = PromptTemplate(
    input_variables=[
        "role",
        "experience",
        "question",
        "answer",

        "confidence_score",
        "confidence_feedback",

        "communication_score",
        "communication_feedback",

        "correctness_score",
        "correctness_feedback",

        "overall_score",
    ],

    template="""
You are a senior interview coach for Talksy AI.

Your task is to synthesize three independent evaluations of the candidate's
current interview answer into concise, actionable overall feedback.

You are NOT responsible for calculating the overall score.

The overall score has already been calculated by the application.


==================================================
INTERVIEW CONTEXT
==================================================

Role:
{role}

Candidate Experience Level:
{experience}


==================================================
CURRENT QUESTION
==================================================

{question}


==================================================
CANDIDATE ANSWER
==================================================

{answer}


==================================================
CONFIDENCE EVALUATION
==================================================

Score:
{confidence_score}

Feedback:
{confidence_feedback}


==================================================
COMMUNICATION EVALUATION
==================================================

Score:
{communication_score}

Feedback:
{communication_feedback}


==================================================
CORRECTNESS EVALUATION
==================================================

Score:
{correctness_score}

Feedback:
{correctness_feedback}


==================================================
OVERALL SCORE
==================================================

{overall_score}


==================================================
YOUR RESPONSIBILITY
==================================================

Synthesize the three evaluations into one useful piece of interview
feedback.

The feedback should answer:

1. What did the candidate do well?

2. What was the most important weakness in the answer?

3. What should the candidate do differently in future interviews?


==================================================
PRIORITIZATION
==================================================

Prioritize the most important issue based on the evaluation.

Technical correctness generally has the highest priority.

However, if correctness is strong but communication is significantly weak,
the feedback should highlight the communication issue.

Similarly, if correctness and communication are strong but confidence is
significantly weak, mention the confidence issue.

Do not mechanically mention all three dimensions if doing so makes the
feedback repetitive.

Focus on the most useful insights.


==================================================
FEEDBACK SYNTHESIS
==================================================

Do NOT simply concatenate:

Confidence feedback +
Communication feedback +
Correctness feedback

Instead, synthesize them into a coherent assessment.

For example, instead of:

"Your confidence was good. Your communication was good. Your correctness
was 6."

Prefer:

"Your core approach was correct and you explained it clearly. However,
the answer missed an important edge case. In future, explicitly discuss
that case before concluding."


==================================================
IMPORTANT RULES
==================================================

- Do not generate a new score.
- Do not modify the provided overall score.
- Do not contradict the individual evaluator results.
- Do not introduce technical claims that are not supported by the
  correctness evaluation.
- Do not overemphasize confidence when correctness is poor.
- Do not penalize concise answers if they adequately answer the question.
- Keep the feedback concise.
- Make the feedback actionable.
- Write feedback directly for the candidate.


==================================================
OUTPUT
==================================================

Return ONLY the structured OverallEvaluation output.
"""
)

In [42]:
SUMMARY_GENERATION_PROMPT = PromptTemplate(
    input_variables=[

        # Previous memory
        "prev_summary",

        # Current interaction
        "cur_question",
        "cur_answer",

        # 3C evaluation
        "confidence_score",
        "confidence_feedback",

        "communication_score",
        "communication_feedback",

        "correctness_score",
        "correctness_feedback",

        # Overall evaluation
        "overall_score",
        "overall_feedback",

        # Strategy
        "follow_up_allowed",
        "follow_up_context",
        "follow_up_cnt",
        "follow_up_score",

        "recent_topic",
        "topic_coverage",

        # Next question
        "next_question",
    ],

    template="""
You are the Interview Memory Agent for Talksy AI.

Your responsibility is to update the compact memory of an ongoing mock
interview.

You are NOT generating the next question.

You are NOT re-evaluating the candidate.

You are NOT writing a transcript.

You are creating a compact decision-making memory that future interview
agents will use to decide:

- whether to ask a follow-up
- whether to introduce a new topic
- whether to increase or decrease difficulty
- which weaknesses to investigate
- which strengths to test further
- which topics have already received sufficient coverage


==================================================
PREVIOUS INTERVIEW MEMORY
==================================================

{prev_summary}


==================================================
CURRENT QUESTION
==================================================

{cur_question}


==================================================
CURRENT ANSWER
==================================================

{cur_answer}


==================================================
CURRENT 3C EVALUATION
==================================================

CONFIDENCE
Score:
{confidence_score}

Feedback:
{confidence_feedback}


COMMUNICATION
Score:
{communication_score}

Feedback:
{communication_feedback}


CORRECTNESS
Score:
{correctness_score}

Feedback:
{correctness_feedback}


==================================================
OVERALL EVALUATION
==================================================

Overall Score:
{overall_score}

Overall Feedback:
{overall_feedback}


==================================================
CURRENT INTERVIEW STRATEGY
==================================================

Follow-up Allowed:
{follow_up_allowed}

Follow-up Context:
{follow_up_context}

Consecutive Follow-up Count:
{follow_up_cnt}

Follow-up Score:
{follow_up_score}

Recent Topic:
{recent_topic}

Existing Topic Coverage:
{topic_coverage}


==================================================
NEXT QUESTION STRATEGY / GENERATED QUESTION
==================================================

Next Question:
{next_question}


IMPORTANT:

The next question is included only to understand the interview direction.

Do NOT use the next question as evidence of candidate performance.

Candidate performance must be inferred from the actual question, answer,
and evaluations.


==================================================
MEMORY UPDATE REQUIREMENTS
==================================================

Update the previous interview memory using the current interaction.

Your summary MUST preserve useful historical information while adding
new evidence.

The summary should contain the following sections:


--------------------------------------------------
1. PERFORMANCE
--------------------------------------------------

Describe the candidate's current overall performance and important
historical trend.

Examples:

- Strong and improving
- Stable around intermediate level
- Strong fundamentals but weak advanced reasoning
- Performance declining on higher difficulty questions


--------------------------------------------------
2. STRENGTHS
--------------------------------------------------

Record meaningful strengths supported by the interview.

Examples:

- Strong backend fundamentals
- Good database knowledge
- Clear explanations
- Strong problem decomposition
- High confidence

Do NOT classify a candidate as having a permanent strength based only
on one unusual answer.


--------------------------------------------------
3. WEAKNESSES
--------------------------------------------------

Record recurring or meaningful weaknesses.

Examples:

- Weak system design depth
- Difficulty handling edge cases
- Incomplete understanding of distributed caching
- Rambling explanations

Distinguish between:

ONE-TIME ISSUE

and

RECURRING WEAKNESS.


--------------------------------------------------
4. TOPICS COVERED
--------------------------------------------------

Maintain a compact list of meaningful topics already explored.

Examples:

- Redis
- MongoDB
- Indexing
- REST APIs
- Caching
- Authentication


--------------------------------------------------
5. RECENT TOPICS
--------------------------------------------------

Track the topics discussed most recently.

This is especially important for preventing excessive consecutive
follow-up questions about one topic.


--------------------------------------------------
6. FOLLOW-UP HISTORY
--------------------------------------------------

Record useful information about follow-up behavior.

For example:

- Redis has been followed up twice.
- Candidate introduced Redis in the current answer.
- Current topic has received sufficient exploration.

Do NOT allow the memory to become dominated by follow-up details.


--------------------------------------------------
7. DIFFICULTY / PERFORMANCE
--------------------------------------------------

Record useful observations about how the candidate performs at different
difficulty levels.

Examples:

- Strong at easy and medium questions.
- Struggles with hard system-design questions.
- Performance remains stable when difficulty increases.


--------------------------------------------------
8. IMPROVEMENT AREAS
--------------------------------------------------

Record areas that deserve future assessment or practice.

Examples:

- System design
- Edge-case reasoning
- Database internals
- Concise technical explanations


--------------------------------------------------
9. FUTURE ASSESSMENT OPPORTUNITIES
--------------------------------------------------

Identify useful directions for future questions.

Examples:

- Explore database internals
- Test distributed-system reasoning
- Assess system-design fundamentals
- Introduce a new topic after repeated Redis follow-ups


==================================================
TOPIC DIVERSITY RULE
==================================================

The interview must maintain balanced topic coverage.

Do NOT allow the summary to encourage infinite follow-ups on one topic.

If a topic has been explored repeatedly, explicitly record that it has
received substantial coverage.

For example:

"Redis/caching has been explored extensively; prefer a new topic unless
a particularly important unresolved weakness remains."

The next strategy agent will use this information to reduce repeated
follow-ups.


==================================================
TREND ANALYSIS
==================================================

Use historical information when available.

Do NOT determine a performance trend from one answer alone.

For example:

Previous scores:
6.2 → 6.8 → 7.4

This supports:

"Performance is improving."

But:

Previous:
8.0
Current:
5.5

does NOT automatically mean:

"Candidate performance is declining."

Instead, consider whether the current question was significantly harder.


==================================================
SUMMARY COMPRESSION
==================================================

The summary will be passed to the next interview turn.

Therefore:

- remove unnecessary details
- remove repeated explanations
- preserve important evidence
- preserve topic history
- preserve weaknesses
- preserve strengths
- preserve performance trends

Do NOT create a transcript.

Do NOT repeat the exact candidate answer unless a specific claim is
important for future reasoning.

Do NOT include evaluator reasoning verbatim.

Do NOT include internal system instructions.


==================================================
SUMMARY FORMAT
==================================================

Use exactly this high-level format:

Performance:
...

Strengths:
- ...
- ...

Weaknesses:
- ...
- ...

Topics Covered:
- ...
- ...

Recent Topics:
- ...
- ...

Follow-up History:
...

Difficulty / Performance:
...

Improvement Areas:
- ...
- ...

Future Assessment Opportunities:
- ...
- ...


==================================================
IMPORTANT
==================================================

The previous summary is the existing memory.

Do NOT discard useful historical information.

Do NOT blindly append the current information.

Instead:

PREVIOUS MEMORY
+
NEW EVIDENCE
=
UPDATED MEMORY


If the new evidence contradicts an old observation, update the observation
instead of keeping both contradictory statements.

Return ONLY the structured SummaryOutput.
"""
)

In [55]:
def format_question(question: Question | None) -> str:

    if question is None:
        return "No previous question. This is the first interview question."

    return f"""
    Question: {question["question"]}
    Difficulty: {question["difficulty"]}
    Type: {question["questionType"]}
    Time Limit: {question["timeLimit"]} seconds
    """.strip()

In [44]:
MAX_CONSECUTIVE_FOLLOW_UPS = 2


def check_follow_up(
    state: InterviewState,
) -> dict:

    strategy_llm = llm.with_structured_output(
        FollowUpDecision
    )

    prompt = CHECK_FOLLOW_UP_PROMPT.format(
        role=state["role"],
        experience=state["experience"],
        mode=state["mode"],

        skills=state["skills"],
        projects=state["projects"],
        exp=state["exp"],
        resumeText=state["resumeText"],

        prev_summary=state["prev_summary"],

        cur_question=format_question(
            state["cur_question"]
        ),

        cur_answer=state["cur_answer"],

        follow_up_allowed=state["follow_up_allowed"],
        follow_up_context=state["follow_up_context"],
        follow_up_cnt=state["follow_up_cnt"],
        follow_up_score=state["follow_up_score"],

        recent_topic=state["recent_topic"],
        topic_coverage=state["topic_coverage"],
    )

    result: FollowUpDecision = strategy_llm.invoke(
        prompt
    )

    follow_up_allowed = result.follow_up_allowed

    # Hard application-level constraint.
    # Do not let the LLM create an endless chain
    # of follow-up questions.
    if state["follow_up_cnt"] >= MAX_CONSECUTIVE_FOLLOW_UPS:
        follow_up_allowed = False

    return {

        "follow_up_allowed": follow_up_allowed,

        "follow_up_context": (
            result.follow_up_context
            if follow_up_allowed
            else ""
        ),

        "follow_up_score": result.follow_up_score,

        "recent_topic": result.detected_topic,

        "next_focus": (
            result.next_focus
            if follow_up_allowed
            else "new_topic"
        ),

        "next_topic": result.next_topic,

        "next_difficulty": result.difficulty,

        "next_question_type": result.question_type,
    }

In [56]:
def generate_question(
    state: InterviewState,
) -> dict:

    question_llm = llm.with_structured_output(
        QuestionOutput
    )
    prev_summary = (
    state["prev_summary"]
    if state["prev_summary"]
    else "No previous interview history. This is the first question."
)
    prompt = GENERATE_QUESTION_PROMPT.format(

        # Interview
        role=state["role"],
        experience=state["experience"],
        mode=state["mode"],

        # Candidate
        skills=state["skills"],
        projects=state["projects"],
        exp=state["exp"],
        resumeText=state["resumeText"],

        # Memory
        prev_summary=prev_summary,

        # Current interaction
        cur_question=format_question(
            state["cur_question"]
        ),
        cur_answer=state["cur_answer"],

        # Follow-up
        follow_up_allowed=state["follow_up_allowed"],
        follow_up_context=state["follow_up_context"],
        follow_up_cnt=state["follow_up_cnt"],
        follow_up_score=state["follow_up_score"],

        # Topic
        recent_topic=state["recent_topic"],
        topic_coverage=state["topic_coverage"],

        # Strategy
        next_focus=state["next_focus"],
        next_topic=state["next_topic"],
        next_difficulty=state["next_difficulty"],
        next_question_type=state["next_question_type"],
    )

    result: QuestionOutput = question_llm.invoke(
        prompt
    )

    return {
        "next_question": {
            "question": result.question,

            # These are controlled by the strategy agent.
            "difficulty": state["next_difficulty"],
            "questionType": state["next_question_type"],

            # Generated by the question generator.
            "timeLimit": result.timeLimit,
        }
    }

In [46]:
def confidence_feedback(
    state: InterviewState,
) -> dict:

    confidence_llm = llm.with_structured_output(
        ConfidenceEvaluation
    )

    prompt = CONFIDENCE_FEEDBACK_PROMPT.format(
        role=state["role"],

        experience=state["experience"],

        question=format_question(
            state["cur_question"]
        ),

        answer=state["cur_answer"],
    )

    result: ConfidenceEvaluation = (
        confidence_llm.invoke(prompt)
    )

    return {
        "confidence_feedback": result.feedback,
        "confidence_score": result.score,
    }

In [47]:
def communication_feedback(
    state: InterviewState,
) -> dict:

    communication_llm = llm.with_structured_output(
        CommunicationEvaluation
    )

    prompt = COMMUNICATION_FEEDBACK_PROMPT.format(
        role=state["role"],

        experience=state["experience"],

        question=format_question(
            state["cur_question"]
        ),

        answer=state["cur_answer"],
    )

    result: CommunicationEvaluation = (
        communication_llm.invoke(prompt)
    )

    return {
        "communication_feedback": result.feedback,
        "communication_score": result.score,
    }

In [48]:
def correctness_feedback(
    state: InterviewState,
) -> dict:

    correctness_llm = llm.with_structured_output(
        CorrectnessEvaluation
    )

    prompt = CORRECTNESS_FEEDBACK_PROMPT.format(

        # Interview
        role=state["role"],
        experience=state["experience"],
        mode=state["mode"],

        # Candidate context
        skills=state["skills"],
        projects=state["projects"],
        exp=state["exp"],
        resumeText=state["resumeText"],

        # Current interaction
        question=format_question(
            state["cur_question"]
        ),
        answer=state["cur_answer"],
    )

    result: CorrectnessEvaluation = (
        correctness_llm.invoke(prompt)
    )

    return {
        "correctness_feedback": result.feedback,
        "correctness_score": result.score,
    }

In [49]:
def overall_feedback(
    state: InterviewState,
) -> dict:

    # -----------------------------------------------
    # 1. Calculate deterministic overall score
    # -----------------------------------------------

    overall_score = round(
        (
            state["correctness_score"] * 0.50
            + state["communication_score"] * 0.30
            + state["confidence_score"] * 0.20
        ),
        2,
    )


    # -----------------------------------------------
    # 2. Generate overall feedback
    # -----------------------------------------------

    overall_llm = llm.with_structured_output(
        OverallEvaluation
    )

    prompt = OVERALL_FEEDBACK_PROMPT.format(

        role=state["role"],

        experience=state["experience"],

        question=format_question(
            state["cur_question"]
        ),

        answer=state["cur_answer"],

        # Confidence
        confidence_score=state["confidence_score"],
        confidence_feedback=state["confidence_feedback"],

        # Communication
        communication_score=state["communication_score"],
        communication_feedback=(
            state["communication_feedback"]
        ),

        # Correctness
        correctness_score=state["correctness_score"],
        correctness_feedback=(
            state["correctness_feedback"]
        ),

        # Deterministic score
        overall_score=overall_score,
    )


    result: OverallEvaluation = (
        overall_llm.invoke(prompt)
    )


    # -----------------------------------------------
    # 3. Update state
    # -----------------------------------------------

    return {
        "overall_feedback": result.feedback,
        "overall_score": overall_score,
    }

In [50]:
def summary_generation(
    state: InterviewState,
) -> dict:

    summary_llm = llm.with_structured_output(
        SummaryOutput
    )

    prompt = SUMMARY_GENERATION_PROMPT.format(

        # Previous memory
        prev_summary=state["prev_summary"],

        # Current interaction
        cur_question=format_question(
            state["cur_question"]
        ),
        cur_answer=state["cur_answer"],

        # Confidence
        confidence_score=state["confidence_score"],
        confidence_feedback=state["confidence_feedback"],

        # Communication
        communication_score=state["communication_score"],
        communication_feedback=(
            state["communication_feedback"]
        ),

        # Correctness
        correctness_score=state["correctness_score"],
        correctness_feedback=(
            state["correctness_feedback"]
        ),

        # Overall
        overall_score=state["overall_score"],
        overall_feedback=state["overall_feedback"],

        # Strategy
        follow_up_allowed=state["follow_up_allowed"],
        follow_up_context=state["follow_up_context"],
        follow_up_cnt=state["follow_up_cnt"],
        follow_up_score=state["follow_up_score"],

        recent_topic=state["recent_topic"],
        topic_coverage=state["topic_coverage"],

        # Next question
        next_question=format_question(
            state["next_question"]
        ),
    )

    result: SummaryOutput = (
        summary_llm.invoke(prompt)
    )

    return {
        "summary": result.summary
    }

In [51]:
start_graph = StateGraph(InterviewState)

start_graph.add_node(
    "generate_question",
    generate_question
)

start_graph.add_edge(
    START,
    "generate_question"
)

start_graph.add_edge(
    "generate_question",
    END
)

start_workflow = start_graph.compile()

In [52]:
answer_graph = StateGraph(InterviewState)


answer_graph.add_node(
    "check_follow_up",
    check_follow_up
)

answer_graph.add_node(
    "generate_question",
    generate_question
)

answer_graph.add_node(
    "confidence_feedback",
    confidence_feedback
)

answer_graph.add_node(
    "communication_feedback",
    communication_feedback
)

answer_graph.add_node(
    "correctness_feedback",
    correctness_feedback
)

answer_graph.add_node(
    "overall_feedback",
    overall_feedback
)

answer_graph.add_node(
    "summary_generation",
    summary_generation
)

In [53]:
# ============================================
# Parallel Strategy + Evaluation
# ============================================

answer_graph.add_edge(
    START,
    "check_follow_up"
)

answer_graph.add_edge(
    START,
    "confidence_feedback"
)

answer_graph.add_edge(
    START,
    "communication_feedback"
)

answer_graph.add_edge(
    START,
    "correctness_feedback"
)


# ============================================
# Question
# ============================================

answer_graph.add_edge(
    "check_follow_up",
    "generate_question"
)


# ============================================
# 3C
# ============================================

answer_graph.add_edge(
    "confidence_feedback",
    "overall_feedback"
)

answer_graph.add_edge(
    "communication_feedback",
    "overall_feedback"
)

answer_graph.add_edge(
    "correctness_feedback",
    "overall_feedback"
)


# ============================================
# Summary synchronization
# ============================================

answer_graph.add_edge(
    "generate_question",
    "summary_generation"
)

answer_graph.add_edge(
    "overall_feedback",
    "summary_generation"
)


# ============================================
# End
# ============================================

answer_graph.add_edge(
    "summary_generation",
    END
)


answer_workflow = answer_graph.compile()

In [58]:
initial_state: InterviewState = {

    # ==================================================
    # Interview Configuration
    # ==================================================

    "role": "Backend Developer",

    "experience": "fresher",

    "mode": "technical",


    # ==================================================
    # Candidate Context
    # ==================================================

    "skills": """
    C++, Python, JavaScript, Node.js, Express.js,
    MongoDB, Redis, REST APIs, Git, Docker
    """,

    "projects": """
    Talksy AI:
    Full-stack AI mock interview platform using React,
    Node.js, Express, MongoDB and LLMs.

    Invoice Aggregator:
    Backend system using Node.js, Express and MongoDB
    for collecting invoices from multiple vendors.
    """,

    "exp": """
    B.Tech Computer Science student with experience
    building full-stack and backend applications.
    """,

    "resumeText": """
    Computer Science engineering student with experience
    in backend development, REST APIs, MongoDB, Redis,
    Node.js, React and AI-powered applications.
    """,


    # ==================================================
    # Interview Memory
    # ==================================================

    "prev_summary": None,


    # ==================================================
    # Current Q/A
    # ==================================================

    "cur_question": None,

    "cur_answer": None,

    "cur_feedback": None,


    # ==================================================
    # Follow-up / Strategy
    # ==================================================

    # No previous question exists
    "follow_up_allowed": False,

    "follow_up_context": "",

    "follow_up_cnt": 0,

    "follow_up_score": 0.0,

    "recent_topic": "",

    "topic_coverage": "",


    # ==================================================
    # Next Question
    # ==================================================

    "next_question": None,


    # ==================================================
    # Current 3C Evaluation
    # ==================================================

    "confidence_feedback": None,

    "communication_feedback": None,

    "correctness_feedback": None,

    "confidence_score": None,

    "communication_score": None,

    "correctness_score": None,


    # ==================================================
    # Overall Evaluation
    # ==================================================

    "overall_feedback": None,

    "overall_score": None,


    # ==================================================
    # Updated Memory
    # ==================================================

    "summary": None,


    # ==================================================
    # First Question Strategy
    # ==================================================

    "next_focus": "initial_assessment",

    "next_topic": "core backend",

    "next_difficulty": "easy",

    "next_question_type": "technical",
}

In [59]:
result = start_workflow.invoke(
    initial_state
)

KeyError: 'next_focus'